# Sold Price vs Listed Price — By Finish (Nonfoil / Foil / Etched)

**Research question:** How do `sold_avg_cents` and `list_low_cents` compare across finish types? Does the sold/list relationship differ for foil vs nonfoil vs etched foil?

**Why it matters:** Foil and etched variants have thinner markets and more volatile pricing. Understanding whether list price is a reliable proxy for sold price — and by how much the relationship changes per finish — is essential for accurate collection valuation.

**Scope:** expansion sets, rare + mythic, all three finishes: NONFOIL, FOIL, ETCHED.

In [1]:
import os
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats
from scipy.stats import linregress

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PARQUET = DATA_DIR / "sold_vs_list_by_finish.parquet"
REFRESH = False  # set True to re-query

FINISHES      = ["nonfoil", "foil", "etched"]
FINISH_CODES  = {"nonfoil": "NONFOIL", "foil": "FOIL", "etched": "ETCHED"}
FINISH_COLORS = {"nonfoil": "#4e79a7", "foil": "#e15759", "etched": "#59a14f"}
RARITY_ORDER  = ["mythic", "rare"]
RARITY_COLORS = {"mythic": "#e07b39", "rare": "#c5a800"}

DB_CONFIG = dict(
    host="localhost",
    port=5433,
    dbname="automana",
    user="automana_admin",
    password=os.environ.get("AUTOMANA_DB_PASSWORD", ""),
)

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def query_to_df(sql):
    with get_conn() as conn:
        cur = conn.cursor()
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

## Part 0 — Coverage by Finish

How often does `sold_avg_cents` exist for each finish type? Foil and etched typically have fewer sales, so coverage should be lower.

In [ ]:
SQL_COVERAGE = """
SELECT
    cf.code                                                          AS finish,
    r.rarity_name,
    COUNT(*)                                                         AS total_rows,
    COUNT(*) FILTER (WHERE ppd.sold_avg_cents IS NOT NULL AND ppd.sold_avg_cents > 0) AS has_sold,
    COUNT(*) FILTER (WHERE ppd.list_low_cents  IS NOT NULL AND ppd.list_low_cents  > 0) AS has_list
FROM pricing.print_price_daily ppd
JOIN card_catalog.card_version cv       ON cv.card_version_id = ppd.card_version_id
JOIN card_catalog.card_finished cf      ON cf.finish_id = ppd.finish_id
JOIN card_catalog.rarities_ref r        ON r.rarity_id = cv.rarity_id
JOIN card_catalog.sets s                ON s.set_id = cv.set_id
JOIN card_catalog.set_type_list_ref st  ON st.set_type_id = s.set_type_id
WHERE st.set_type = 'expansion'
  AND cf.code IN ('NONFOIL', 'FOIL', 'ETCHED')
  AND r.rarity_name IN ('rare', 'mythic')
GROUP BY cf.code, r.rarity_name
ORDER BY cf.code, r.rarity_name
"""

cov = query_to_df(SQL_COVERAGE)
cov["sold_pct"] = (cov["has_sold"] / cov["total_rows"] * 100).round(1)
cov["list_pct"] = (cov["has_list"] / cov["total_rows"] * 100).round(1)
display(cov[["finish", "rarity_name", "total_rows", "has_sold", "sold_pct", "has_list", "list_pct"]])

In [ ]:
# Coverage bar chart — sold vs list per finish
pivot = cov.pivot_table(index="finish", columns="rarity_name", values="sold_pct")
ax = pivot.plot(kind="bar", figsize=(10, 5), color=[RARITY_COLORS[r] for r in pivot.columns], edgecolor="white")
ax.set_xlabel("Finish")
ax.set_ylabel("% rows with sold_avg_cents")
ax.set_title("Sold price coverage by finish and rarity")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Rarity")
plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig0_coverage.png", dpi=150)
plt.show()

## Part 1 — Load rows where both prices exist

Restrict to rows where **both** `list_low_cents` and `sold_avg_cents` are present and positive — the overlap we can actually compare.

In [ ]:
SQL = """
SELECT
    lower(cf.code)           AS finish,
    r.rarity_name,
    cv.card_version_id,
    ppd.price_date,
    ppd.list_low_cents,
    ppd.list_avg_cents,
    ppd.sold_avg_cents
FROM pricing.print_price_daily ppd
JOIN card_catalog.card_version cv       ON cv.card_version_id = ppd.card_version_id
JOIN card_catalog.card_finished cf      ON cf.finish_id = ppd.finish_id
JOIN card_catalog.rarities_ref r        ON r.rarity_id = cv.rarity_id
JOIN card_catalog.sets s                ON s.set_id = cv.set_id
JOIN card_catalog.set_type_list_ref st  ON st.set_type_id = s.set_type_id
WHERE st.set_type = 'expansion'
  AND cf.code IN ('NONFOIL', 'FOIL', 'ETCHED')
  AND ppd.list_low_cents  IS NOT NULL AND ppd.list_low_cents  > 0
  AND ppd.sold_avg_cents  IS NOT NULL AND ppd.sold_avg_cents  > 0
  AND r.rarity_name IN ('rare', 'mythic')
"""

if REFRESH or not PARQUET.exists():
    print("Querying DB…")
    df = query_to_df(SQL)
    df["price_date"] = pd.to_datetime(df["price_date"])
    df.to_parquet(PARQUET, index=False)
    print(f"Saved {len(df):,} rows → {PARQUET}")
else:
    df = pd.read_parquet(PARQUET)
    print(f"Loaded {len(df):,} rows from {PARQUET}")

df["list_low"]  = df["list_low_cents"]  / 100
df["list_avg"]  = df["list_avg_cents"]  / 100
df["sold_avg"]  = df["sold_avg_cents"]  / 100
df["sold_list_ratio"] = df["sold_avg"] / df["list_low"]

print("\nRows by finish × rarity:")
print(df.groupby(["finish", "rarity_name"]).size().unstack(fill_value=0))

## Part 2 — Price distribution by finish

How do list and sold prices compare within each finish? We expect foil/etched to skew higher and have wider spreads.

In [ ]:
stats_rows = []
for finish in FINISHES:
    sub = df[df["finish"] == finish]
    if sub.empty:
        continue
    for col, label in [("list_low", "list_low"), ("sold_avg", "sold_avg")]:
        q = sub[col].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
        stats_rows.append({
            "finish": finish, "price_type": label,
            "n": len(sub),
            "p10": q[0.1], "p25": q[0.25], "median": q[0.5],
            "p75": q[0.75], "p90": q[0.9], "mean": sub[col].mean(),
        })

stats_df = pd.DataFrame(stats_rows).set_index(["finish", "price_type"])
display(stats_df.round(2))

In [ ]:
bins = np.logspace(np.log10(0.05), np.log10(500), 80)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, finish in zip(axes, FINISHES):
    sub = df[df["finish"] == finish]
    if sub.empty:
        ax.set_title(f"{finish.capitalize()} — no data")
        continue
    ax.hist(sub["list_low"].clip(upper=500), bins=bins, alpha=0.55, color="#4e79a7", label="list_low")
    ax.hist(sub["sold_avg"].clip(upper=500), bins=bins, alpha=0.55, color="#e15759", label="sold_avg")
    ax.set_xscale("log")
    ax.set_xlabel("Price (USD, log scale)")
    ax.set_ylabel("Row count")
    ax.set_title(f"{finish.capitalize()} — list_low vs sold_avg (n={len(sub):,})")
    ax.legend()
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())

plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig1_distributions.png", dpi=150)
plt.show()

## Part 3 — Correlation by finish

We compute:
- **Pearson** on log-transformed prices
- **Spearman rank** correlation (robust to outliers)

The question: does the list→sold correlation hold as well for foil and etched as for nonfoil?

In [ ]:
corr_rows = []
for finish in FINISHES:
    sub = df[df["finish"] == finish].dropna(subset=["list_low", "sold_avg"])
    if len(sub) < 100:
        print(f"{finish}: insufficient data ({len(sub)} rows)")
        continue
    log_list = np.log(sub["list_low"])
    log_sold = np.log(sub["sold_avg"])
    pearson_r, _ = stats.pearsonr(log_list, log_sold)
    spearman_r, _ = stats.spearmanr(sub["list_low"], sub["sold_avg"])
    corr_rows.append({"finish": finish, "n": len(sub), "pearson_log": pearson_r, "spearman": spearman_r})
    print(f"\n{'='*50}")
    print(f"{finish.upper()}  (n={len(sub):,})")
    print(f"  Pearson  (log-log): r = {pearson_r:.4f}")
    print(f"  Spearman (rank):    r = {spearman_r:.4f}")

corr_df = pd.DataFrame(corr_rows).set_index("finish")
display(corr_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(corr_df))
width = 0.35
ax.bar(x - width/2, corr_df["pearson_log"], width, label="Pearson (log-log)", color="#4e79a7")
ax.bar(x + width/2, corr_df["spearman"],    width, label="Spearman (rank)",   color="#e15759")
ax.set_xticks(x)
ax.set_xticklabels(corr_df.index)
ax.set_ylim(0.8, 1.01)
ax.set_ylabel("Correlation coefficient")
ax.set_title("Sold vs List price correlation by finish")
ax.legend()
ax.axhline(0.95, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig2_correlation.png", dpi=150)
plt.show()

## Part 4 — Sold / List ratio by finish

If `sold_avg = k × list_low`, we can infer one from the other with a multiplier. The ratio may differ significantly between nonfoil, foil, and etched — and may vary by price level within each finish.

In [ ]:
df_clean = df[(df["sold_list_ratio"] >= 0.1) & (df["sold_list_ratio"] <= 10)].copy()

print("sold_avg / list_low ratio distribution by finish:")
ratio_stats = df_clean.groupby("finish")["sold_list_ratio"].describe().round(3)
display(ratio_stats)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, finish in zip(axes, FINISHES):
    sub = df_clean[df_clean["finish"] == finish]["sold_list_ratio"]
    if sub.empty:
        ax.set_title(f"{finish.capitalize()} — no data")
        continue
    med = sub.median()
    ax.hist(sub, bins=80, color=FINISH_COLORS[finish], alpha=0.85, edgecolor="white")
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1, label="ratio = 1 (sold = list)")
    ax.axvline(med, color="red", linestyle="-", linewidth=1.5, label=f"median = {med:.2f}")
    ax.set_xlabel("sold_avg / list_low")
    ax.set_ylabel("Row count")
    ax.set_title(f"{finish.capitalize()} — sold/list ratio (n={len(sub):,})")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig3_ratio_dist.png", dpi=150)
plt.show()

In [ ]:
PRICE_BINS   = [0, 0.5, 1, 2, 5, 10, 25, 50, 100, 500]
PRICE_LABELS = ["<$0.50", "$0.50-1", "$1-2", "$2-5", "$5-10", "$10-25", "$25-50", "$50-100", ">$100"]

df_clean["price_bucket"] = pd.cut(df_clean["list_low"], bins=PRICE_BINS, labels=PRICE_LABELS)

fig, ax = plt.subplots(figsize=(13, 5))
for finish in FINISHES:
    sub = df_clean[df_clean["finish"] == finish]
    if sub.empty:
        continue
    ratio_by_bucket = (
        sub.groupby("price_bucket", observed=True)["sold_list_ratio"]
        .agg(["median", "count"])
    )
    valid = ratio_by_bucket[ratio_by_bucket["count"] >= 50].reset_index()
    ax.plot(valid["price_bucket"].astype(str), valid["median"],
            marker="o", linewidth=2, color=FINISH_COLORS[finish], label=finish.capitalize())

ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.5, label="ratio = 1")
ax.set_xlabel("list_low price bracket")
ax.set_ylabel("Median sold_avg / list_low")
ax.set_title("Does the sold/list ratio vary by price level? (per finish)")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig4_ratio_by_price.png", dpi=150)
plt.show()

## Part 5 — Log-log regression by finish

We fit `log(sold_avg) = a + b × log(list_low)` per finish.

- **b ≈ 1**: sold ≈ constant × list — a simple multiplier suffices
- **b < 1**: sold prices are compressed at the high end vs list
- **R²**: how much of the variance in sold price is explained by list price alone

In [ ]:
SAMPLE = 30_000

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, finish in zip(axes, FINISHES):
    sub = df_clean[df_clean["finish"] == finish]
    if len(sub) < 100:
        ax.set_title(f"{finish.capitalize()} — insufficient data")
        continue
    samp = sub.sample(min(SAMPLE, len(sub)), random_state=42)
    log_x = np.log(samp["list_low"])
    log_y = np.log(samp["sold_avg"])
    slope, intercept, r, _, _ = linregress(log_x, log_y)
    r2 = r ** 2

    x_range = np.linspace(log_x.min(), log_x.max(), 200)
    y_fit   = intercept + slope * x_range

    ax.scatter(np.exp(log_x), np.exp(log_y), alpha=0.06, s=3, color=FINISH_COLORS[finish])
    ax.plot(np.exp(x_range), np.exp(y_fit), color="black", linewidth=2,
            label=f"slope={slope:.3f}, R²={r2:.3f}")
    ax.plot([0.05, 300], [0.05, 300], "--", color="grey", linewidth=1, alpha=0.5, label="y = x")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(0.05, 300)
    ax.set_ylim(0.05, 300)
    ax.set_xlabel("list_low (USD)")
    ax.set_ylabel("sold_avg (USD)")
    ax.set_title(f"{finish.capitalize()} — log-log regression")
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

    print(f"\n{finish.upper()}")
    print(f"  log(sold) = {intercept:.3f} + {slope:.3f} × log(list_low)")
    print(f"  R² = {r2:.4f}  → list_low explains {r2*100:.1f}% of variance in sold_avg")
    print(f"  Implied multiplier at list=$1:  sold ≈ ${np.exp(intercept):.2f}")
    print(f"  Implied multiplier at list=$10: sold ≈ ${np.exp(intercept + slope*np.log(10)):.2f}")
    print(f"  Implied multiplier at list=$50: sold ≈ ${np.exp(intercept + slope*np.log(50)):.2f}")

plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig5_regression.png", dpi=150)
plt.show()

## Part 6 — Cross-finish comparison

How does the median price level differ between nonfoil, foil, and etched for the same cards? The **foil multiplier** (foil price / nonfoil price) is a key MTG finance metric.

In [ ]:
# For each card_version_id × price_date, compare across finishes
# Use the most recent 90 days to keep it current
recent = df[df["price_date"] >= df["price_date"].max() - pd.Timedelta(days=90)].copy()

# Pivot: one row per card_version_id × price_date, columns = finish prices
pivot = recent.pivot_table(
    index=["card_version_id", "price_date", "rarity_name"],
    columns="finish",
    values="sold_avg",
    aggfunc="first"
).reset_index()

for col in ["nonfoil", "foil", "etched"]:
    if col not in pivot.columns:
        pivot[col] = np.nan

# Foil multiplier = foil sold / nonfoil sold
pivot_paired = pivot.dropna(subset=["nonfoil", "foil"]).copy()
pivot_paired["foil_mult"] = pivot_paired["foil"] / pivot_paired["nonfoil"]
pivot_paired_etch = pivot.dropna(subset=["nonfoil", "etched"]).copy()
pivot_paired_etch["etched_mult"] = pivot_paired_etch["etched"] / pivot_paired_etch["nonfoil"]

print("Foil multiplier (foil sold / nonfoil sold) — last 90 days:")
print(pivot_paired.groupby("rarity_name")["foil_mult"].describe().round(2))

if not pivot_paired_etch.empty:
    print("\nEtched multiplier (etched sold / nonfoil sold) — last 90 days:")
    print(pivot_paired_etch.groupby("rarity_name")["etched_mult"].describe().round(2))

In [ ]:
# Cap multipliers for readability
foil_clean  = pivot_paired[pivot_paired["foil_mult"].between(0.2, 15)]
etch_clean  = pivot_paired_etch[pivot_paired_etch["etched_mult"].between(0.2, 15)] if not pivot_paired_etch.empty else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, color in [
    (axes[0], foil_clean,  "Foil / Nonfoil sold price multiplier",   FINISH_COLORS["foil"]),
    (axes[1], etch_clean, "Etched / Nonfoil sold price multiplier", FINISH_COLORS["etched"]),
]:
    col = "foil_mult" if "foil_mult" in data.columns else "etched_mult"
    if data.empty or col not in data.columns:
        ax.text(0.5, 0.5, "No paired data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        continue
    med = data[col].median()
    ax.hist(data[col], bins=60, color=color, alpha=0.85, edgecolor="white")
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1, label="multiplier = 1")
    ax.axvline(med, color="red", linestyle="-", linewidth=1.5, label=f"median = {med:.2f}×")
    ax.set_xlabel("Price multiplier vs nonfoil")
    ax.set_ylabel("Count")
    ax.set_title(f"{title} (n={len(data):,})")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / "finish_fig6_foil_multiplier.png", dpi=150)
plt.show()

## Part 7 — Summary

Consolidated findings across all three finish types.

In [ ]:
print("SUMMARY — SOLD vs LIST PRICE BY FINISH")
print("=" * 60)
for finish in FINISHES:
    sub = df_clean[df_clean["finish"] == finish]
    if sub.empty:
        print(f"\n{finish.upper()}: no data")
        continue
    ratio = sub["sold_list_ratio"]
    log_x = np.log(sub["list_low"])
    log_y = np.log(sub["sold_avg"])
    slope, intercept, r, _, _ = linregress(log_x, log_y)
    spearman_r, _ = stats.spearmanr(sub["list_low"], sub["sold_avg"])
    print(f"\n{finish.upper()} (n={len(sub):,})")
    print(f"  Spearman rank corr:   {spearman_r:.3f}")
    print(f"  Log-log R²:           {r**2:.3f}")
    print(f"  Median sold/list:     {ratio.median():.3f}")
    print(f"  IQR sold/list:        [{ratio.quantile(0.25):.2f}, {ratio.quantile(0.75):.2f}]")
    print(f"  Regression slope:     {slope:.3f}  (1.0 = pure multiplier)")
    print(f"  Implied sold @ $1 list:  ${np.exp(intercept):.2f}")
    print(f"  Implied sold @ $10 list: ${np.exp(intercept + slope*np.log(10)):.2f}")